# Klassifikation mit verschiedenen Leveln der Personalisierung

1. Subject-independent
2. Subject Baseline Norm - t0 als baseline; t0 mittel wird von allen samples abgezogen
3. Mit Kalibrierungs Samples - t0-t5; 2,4,6 samples pro stufe mit ins training
4. Nur auf Daten einer Person - cv within subject

In [8]:
from pathlib import Path
import numpy as np
import pandas as pd

from scripts.myml import loso_binary_nested_cv, loso_multiclass_nested_cv, loso_binary_calibrated_nested_cv, safe_results_multiclass
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from scripts.feature_engeneering import subject_baseline_normalization
from xgboost import XGBClassifier

### Daten Laden

In [9]:
DATASET_PATH = Path("dataset/np-dataset")
X = np.load(DATASET_PATH / "X.npy")
X_catch22 = np.load(DATASET_PATH / "X_catch22.npy")
X_catch22_feature_names = np.load(DATASET_PATH / "feature_names.npy")
y_heater = np.load(DATASET_PATH / "y_heater.npy")
subjects = np.load(DATASET_PATH / "subjects.npy")
y = np.argmax(y_heater, axis=1)

In [10]:
print(f"X shape : {X_catch22.shape}")
print(f"y shape : {y.shape}   classes: {np.unique(y).tolist()}")
print(f"subjects : {np.unique(subjects).shape[0]} unique subjects")
print(f"Class counts : { {c: int((y==c).sum()) for c in range(6)} }")

X shape : (2495, 174)
y shape : (2495,)   classes: [0, 1, 2, 3, 4, 5]
subjects : 52 unique subjects
Class counts : {0: 416, 1: 416, 2: 416, 3: 415, 4: 416, 5: 416}


### Hyperparameter Grid

In [11]:
# Random Forest
PARAM_GRID_RF = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
}

PARAM_GRID_XGB = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 6, 10],
    'learning_rate': [0.01, 0.1, 0.2],
}

INNER_N_SPLITS = 5
RANDOM_STATE = 42

In [12]:
# setup for comparison of models
comparison_results = []

In [13]:
# without normalization
X_catch22 = X_catch22.copy()

# with subject-specific normalization
baseline_class = 0 # samples with no stimulus 
X_catch22_normalized = subject_baseline_normalization(X_catch22, y, subjects, baseline_class)

print(np.nanmean(X_catch22))
print(np.nanmean(X_catch22_normalized))

40.75209365022363
1.2530141390178788


# Multiclass t0-t5 - Random Forest Classifier

## 1. Subject-norm-baseline

In [14]:
pers_comparison = [X_catch22, X_catch22_normalized]

for X_comp in pers_comparison:
    
    print(f"\nPersonalization: {'Baseline Normalized' if X_comp is X_catch22_normalized else 'No Normalization'}")

    # Define model
    rf_classifier = RandomForestClassifier(random_state=RANDOM_STATE)

    print(f"Training model: {rf_classifier}...")
    results = metrics = loso_multiclass_nested_cv(
        X_comp, 
        y, 
        subjects, 
        rf_classifier, 
        PARAM_GRID_RF, 
        'classifier'
        )
    result = safe_results_multiclass(
        model_name='RandomForest',
        personalization='Normalized' if X_comp is X_catch22_normalized else 'NoNormalization',
        metrics=metrics
    )
    comparison_results.append(result)
    
    
    print(f"    Accuracy : {metrics['accuracy']:.3f} ± {metrics['accuracy_std']:.3f}")
    print(f"    F1       : {metrics['f1']:.3f} ± {metrics['f1_std']:.3f}")
    print(f"    AUC      : {metrics['auc']:.3f} ± {metrics['auc_std']:.3f}")
    print(f"    MAE      : {metrics['mae']:.3f} ± {metrics['mae_std']:.3f}")
    print(f"    RMSE      : {metrics['rmse']:.3f} ± {metrics['rmse_std']:.3f}")


Personalization: No Normalization
Training model: RandomForestClassifier(random_state=42)...


Results saved to test_multiclass_RandomForest_NoNormalization.csv
    Accuracy : 0.320 ± 0.079
    F1       : 0.280 ± 0.071
    AUC      : 0.709 ± 0.071
    MAE      : 1.183 ± 0.248
    RMSE      : 1.602 ± 0.276

Personalization: Baseline Normalized
Training model: RandomForestClassifier(random_state=42)...
Results saved to test_multiclass_RandomForest_Normalized.csv
    Accuracy : 0.401 ± 0.076
    F1       : 0.353 ± 0.074
    AUC      : 0.755 ± 0.061
    MAE      : 1.067 ± 0.235
    RMSE      : 1.539 ± 0.273


Frage hierzu - normalerweise zählt es als data leakage, wenn man erst alles Normalisiert und danach Train/ test splits macht. Hier nutze ich jedoch nur t0 segmente um eine Null baseline zu schaffen, die im klinischen einsatz auch verfügbar wären. Daher okay? - da es als kalibrierung dient?

# 2. Mit Kalibrierung - Random Forest Classifier

# Mutliclass t0-t5 - XGBoost

## 1. Subject-norm-baseline

In [15]:
pers_comparison = [X_catch22, X_catch22_normalized]

for X_comp in pers_comparison:
    
    print(f"\nPersonalization: {'Baseline Normalized' if X_comp is X_catch22_normalized else 'No Normalization'}")

    # Define model
    xgb_classifier = XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss')

    print(f"Training model: {xgb_classifier}...")
    metrics = loso_multiclass_nested_cv(
        X_comp, 
        y, 
        subjects, 
        xgb_classifier, 
        PARAM_GRID_XGB, 
        'classifier'
        )
    result = safe_results_multiclass(
        model_name='XGBClassifier',
        personalization='Normalized' if X_comp is X_catch22_normalized else 'NoNormalization',
        metrics=metrics
    )
    comparison_results.append(result)
    
    print(f"    Accuracy : {metrics['accuracy']:.3f} ± {metrics['accuracy_std']:.3f}")
    print(f"    F1       : {metrics['f1']:.3f} ± {metrics['f1_std']:.3f}")
    print(f"    AUC      : {metrics['auc']:.3f} ± {metrics['auc_std']:.3f}")
    print(f"    MAE      : {metrics['mae']:.3f} ± {metrics['mae_std']:.3f}")
    print(f"    RMSE      : {metrics['rmse']:.3f} ± {metrics['rmse_std']:.3f}")


Personalization: No Normalization
Training model: XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)...


/Users/wielandcremer/Documents/Uni/6. Semester/pain_recognition_ba/painenv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/Users/wielandcremer/Documents/Uni/6. Semester/pain_recognition_ba/painenv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/Users/wielandcremer/Documents/Uni/6. Semester/pain_recognition_ba/painenv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/Users/wielandcremer/Documents/Uni/6. Semester/pain_recognition_

Results saved to test_multiclass_XGBClassifier_NoNormalization.csv
    Accuracy : 0.331 ± 0.072
    F1       : 0.293 ± 0.070
    AUC      : 0.710 ± 0.056
    MAE      : 1.207 ± 0.267
    RMSE      : 1.646 ± 0.289

Personalization: Baseline Normalized
Training model: XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=No

# Direkter vergleich

In [16]:
df_all = pd.DataFrame(comparison_results)
display(df_all.round(3))

,Model,Personalization,K,Accuracy,Accuracy_Std,F1,F1_Std,AUC,AUC_Std,MAE,MAE_Std,RMSE,RMSE_Std
0,RandomForest,NoNormalization,None,0.320,0.079,0.280,0.071,0.709,0.071,1.183,0.248,1.602,0.276
1,RandomForest,Normalized,None,0.401,0.076,0.353,0.074,0.755,0.061,1.067,0.235,1.539,0.273
2,XGBClassifier,NoNormalization,None,0.331,0.072,0.293,0.070,0.710,0.056,1.207,0.267,1.646,0.289
3,XGBClassifier,Normalized,None,0.392,0.078,0.353,0.074,0.744,0.055,1.049,0.234,1.500,0.259
